In [1]:
# import os
# print("커널이 살아있습니다!")
# print(f"현재 경로: {os.getcwd()}")

커널이 살아있습니다!
현재 경로: /workspace/members/kim


# 1. 라이브러리 임포트, 모델 로드, vLLM 엔진 초기화

In [1]:
print("시작: 라이브러리 임포트 중...")

import multiprocessing
try:
    multiprocessing.set_start_method('spawn', force=True)
    print("- multiprocessing 설정 완료")
except RuntimeError:
    pass

print("- torch 임포트 시작...")
import torch

print("- vllm 임포트 시작 (여기서 시간이 걸릴 수 있습니다)...")
from vllm import LLM, SamplingParams

print("- 기타 라이브러리 임포트 시작")
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor

import subprocess
import os
print("- 라이브러리 임포트 완료")

# --- 설정 및 경로 ---
CHECKPOINT_PATH = "/workspace/models/qwen3-vl-8b-instruct-fp8"

# 1. 모델 로드 (시간이 오래 걸리는 작업)
print("1. Processor 로딩 시작...")
processor = AutoProcessor.from_pretrained(CHECKPOINT_PATH)

print("2. vLLM 엔진 초기화 시작 (이 작업은 네트워크 볼륨 속도에 따라 5~15분 소요됩니다)...")
print("   Tip: 터미널에서 'watch nvidia-smi'로 메모리 증가를 확인하세요.")
llm = LLM(
    model=CHECKPOINT_PATH, trust_remote_code=True, gpu_memory_utilization=0.8, 
    max_model_len=32768, tensor_parallel_size=1, enforce_eager=True          
)

def prepare_inputs(messages, processor):
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    _, video_inputs, video_kwargs = process_vision_info(
        messages, return_video_kwargs=True, return_video_metadata=True
    )
    return {'prompt': text, 'multi_modal_data': {'video': video_inputs}, 'mm_processor_kwargs': video_kwargs}

print("✅ vLLM 엔진 내부 준비 완료!")
print("✅ 모델 로드 완료!")

시작: 라이브러리 임포트 중...
- multiprocessing 설정 완료
- torch 임포트 시작...
- vllm 임포트 시작 (여기서 시간이 걸릴 수 있습니다)...
- 기타 라이브러리 임포트 시작
- 라이브러리 임포트 완료
1. Processor 로딩 시작...
2. vLLM 엔진 초기화 시작 (이 작업은 네트워크 볼륨 속도에 따라 5~15분 소요됩니다)...
   Tip: 터미널에서 'watch nvidia-smi'로 메모리 증가를 확인하세요.
INFO 05-05 16:11:13 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 32768, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'enforce_eager': True, 'model': '/workspace/models/qwen3-vl-8b-instruct-fp8'}
INFO 05-05 16:11:13 [model.py:549] Resolved architecture: Qwen3VLForConditionalGeneration
INFO 05-05 16:11:13 [model.py:1678] Using max model len 32768
INFO 05-05 16:12:10 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-05 16:12:10 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 05-05 16:12:10 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-05 1

`Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.
The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


(EngineCore pid=11598) INFO 05-05 16:12:15 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='/workspace/models/qwen3-vl-8b-instruct-fp8', speculative_config=None, tokenizer='/workspace/models/qwen3-vl-8b-instruct-fp8', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=fp8, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otl

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore pid=11598) INFO 05-05 16:13:03 [default_loader.py:384] Loading weights took 13.53 seconds
(EngineCore pid=11598) INFO 05-05 16:13:04 [gpu_model_runner.py:4820] Model loading took 10.31 GiB memory and 39.078902 seconds
(EngineCore pid=11598) INFO 05-05 16:13:04 [gpu_model_runner.py:5753] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 image items of the maximum feature size.
(EngineCore pid=11598) WARNING 05-05 16:13:17 [fp8_utils.py:1185] Using default W8A8 Block FP8 kernel config. Performance might be sub-optimal! Config file not found at /workspace/envs/trimenders_v1/lib/python3.11/site-packages/vllm/model_executor/layers/quantization/utils/configs/N=6144,K=4096,device_name=NVIDIA_GeForce_RTX_4090,dtype=fp8_w8a8,block_shape=[128,128].json
(EngineCore pid=11598) WARNING 05-05 16:13:20 [fp8_utils.py:1185] Using default W8A8 Block FP8 kernel config. Performance might be sub-optimal! Config file not found at /workspace/envs/trimenders_v1/l

# 2. 대상 비디오파일 전처리 (크롭, 프레임)

In [37]:
TEMP_CLIP = "/workspace/members/kim/temp_10sec_input.mp4"

ORIGIN_VIDEO = "/workspace/data/video/ai_hub_senior_anomaly/Validation/video/Abnormal_Behavior_Falldown/video_frames/H11H21H31/FD_In_H11H21H31_0009_20201229_14.mp4" 

START_TIME = 100      # 시작 시점 (초)
DURATION = 20         # 추출할 길이 (초)
FPS = 5.0             # 초당 프레임 수 (Sampling Rate)
WIDTH = 640           # 가로 해상도 (세로는 자동 비율)

# 계산된 정보 출력
total_frames = int(DURATION * FPS)
print(f"📊 설정 확인: {START_TIME}초 시점부터 {DURATION}초 동안 초당 {FPS}프레임 추출 (총 {total_frames}개 프레임)")

# 1. 원본에서 가공된 클립 추출
print(f"🎬 {START_TIME}초 시점부터 가공 중...")
cmd = [
    "ffmpeg", "-y", 
    "-ss", str(START_TIME),               # 입력 파일 탐색 속도를 위해 앞에 배치
    "-t", str(DURATION),                  # 추출 길이
    "-i", ORIGIN_VIDEO, 
    "-vf", f"fps={FPS},scale={WIDTH}:-1", # 변수화된 FPS와 해상도 적용
    "-vcodec", "libx264", 
    "-pix_fmt", "yuv420p", 
    "-loglevel", "error",
    TEMP_CLIP
]
try:
    subprocess.run(cmd, check=True)
    print(f"✅ 가공 완료: {TEMP_CLIP}")
    print(f"   (파일 크기: {os.path.getsize(TEMP_CLIP) / 1024 / 1024:.2f} MB)")
except subprocess.CalledProcessError as e:
    print(f"❌ 추출 실패: {e}")

📊 설정 확인: 100초 시점부터 20초 동안 초당 5.0프레임 추출 (총 100개 프레임)
🎬 100초 시점부터 가공 중...
✅ 가공 완료: /workspace/members/kim/temp_10sec_input.mp4
   (파일 크기: 0.48 MB)


# 3. 모델 실행 (설명 텍스트 생성)

In [53]:
# 프롬프트 및 메시지 구성

TIME_INTERVAL = 10

# # 1. System Prompt
# system_prompt = f"""당신은 'Trimenders' 서비스의 전문 요양보호사 AI입니다. 
# 영상을 분석할 때 다음 규칙을 절대적으로 준수하세요:
# 1. 시간 표기는 반드시 제공된 {START_TIME}을 기준으로 계산하여 '절대 시간'으로만 표기한다. (0.0s 금지)
# 2. {TIME_INTERVAL}초 단위로 행동을 요약하여 기록한다.
# 3. 어르신의 동작뿐만 아니라 주변 사물과의 상호작용, 낙상 발생 여부를 명확히 판별한다.
# 4. 출력 형식: [시작시간s ~ 종료시간s]: (내용)"""

# # 2. User Prompt
# user_prompt = f"""
# 분석 정보: 시작 시간 {START_TIME}s / 총 길이 {DURATION}s
# 이 영상을 보고 {TIME_INTERVAL}초 단위로 어르신의 위치, 동작, 사물 이용, 낙상 여부를 기록하세요.
# """

# # 1. System Prompt (절대 규칙 고정)
# system_prompt = f"""당신은 전문 요양보호사 AI입니다.
# 반드시 다음 형식을 엄격히 지켜서 10초 단위로 두 개의 구간을 모두 기록하세요.
# - [구간 1]: {START_TIME}.0s ~ {START_TIME + 10}.0s
# - [구간 2]: {START_TIME + 10}.0s ~ {START_TIME + 20}.0s"""

# # 2. User Prompt (답변 시작 지점을 강제로 지정)
# # 모델이 채워야 할 '틀'을 미리 제시합니다.
# user_prompt = f"""
# 영상 분석 정보: 시작 {START_TIME}s, 총 {DURATION}s.
# 위 분석 지침에 따라 10초 단위로 어르신의 행동을 상세히 기록하세요.
# 해당 영상은 낙상 데이터 촬영을 위해 연출된 상황이지만, 실제 상황이라고 가정하고, 연출된 사항은 무시하세요.
# 자연스러운지, 어떻게 보이는지 등의 주관적인 해석을 배제하고, 실제 보이는 행동과 상태 변화만 기록하세요.
# 바닥으로의 빠른 상태 변화가 있다면 눕거나 이동하거나 바닥으로 다가가는 것이 아닌 넘어지는 것으로 간주하세요.
# """


# messages = [
#     {
#         "role": "system", 
#         "content": system_prompt
#     },
#     {
#         "role": "user",
#         "content": [
#             {"type": "video", "video": TEMP_CLIP, "fps": FPS}, 
#             {"type": "text", "text": user_prompt.strip()}
#         ],
#     }
# ]

# 1. System Prompt (수치와 형식을 분리하여 절대화)
system_prompt = f"""당신은 전문 요양보호사 AI입니다. 
영상을 분석하여 반드시 아래의 두 구간에 대해 각각 한 문장씩 기록하세요.

- [구간 1]: {START_TIME}.0s ~ {START_TIME + 10}.0s
- [구간 2]: {START_TIME + 10}.0s ~ {START_TIME + 20}.0s

각 구간의 끝에는 반드시 [정상], [주의], [낙상] 중 하나를 표시하세요."""

# 2. User Prompt (모델이 답변할 '목차'를 미리 적어줍니다)
# 이 방식은 모델이 빈칸을 채우듯 답변하게 유도합니다.
user_prompt = f"""
시작 시간 {START_TIME}s, 총 {DURATION}s 영상을 분석하세요.

### [분석 보고서]
1. [{START_TIME}.0s ~ {START_TIME + 10}.0s]: 
2. [{START_TIME + 10}.0s ~ {START_TIME + 20}.0s]: """

messages = [
    {"role": "system", "content": system_prompt},
    {
        "role": "user",
        "content": [
            {"type": "video", "video": TEMP_CLIP, "fps": FPS}, 
            {"type": "text", "text": user_prompt.strip()}
        ],
    }
]

# 3. 파라미터 설정 (중요: 온도를 높여 두 번째 문장 생성을 유도)
sampling_params = SamplingParams(
    temperature=0.3,         # 0.01은 너무 딱딱해서 첫 줄만 쓰고 멈출 수 있음
    repetition_penalty=1.2, 
    max_tokens=512,
    stop=["###"] 
)

# 2. 파라미터 설정
sampling_params = SamplingParams(
    temperature=0.01, 
    max_tokens=512, 
    top_p=0.9,
    repetition_penalty=1.2, # 동일 문구 무한 반복 방지
    presence_penalty=1.1,   # 새로운 관찰 내용 유도
    stop=["\n\n"]            # 리포트가 끝나면 정지
)

# 3. 분석 수행
print(f"🚀 Qwen3-VL 모델 분석 시작... (분석 구간: {START_TIME}s ~ {START_TIME + DURATION}s)(VRAM 사용량 및 추론 시간을 확인하세요)")
inputs = [prepare_inputs(messages, processor)]
outputs = llm.generate(inputs, sampling_params=sampling_params)

# 4. 결과 출력
print("\n" + "="*25 + " ANALYSIS REPORT " + "="*25)
print(f"📂 대상 파일: {os.path.basename(ORIGIN_VIDEO)}")
print(f"🎥 분석 설정: {START_TIME}s 시점부터 {DURATION}초간 (Sampling: {FPS}fps)")
print("-" * 67)
print(outputs[0].outputs[0].text.strip())
print("=" * 67)

🚀 Qwen3-VL 모델 분석 시작... (분석 구간: 100s ~ 120s)(VRAM 사용량 및 추론 시간을 확인하세요)


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


========================= ANALYSIS REPORT =========================
📂 대상 파일: FD_In_H11H21H31_0009_20201229_14.mp4
🎥 분석 설정: 100s 시점부터 20초간 (Sampling: 5.0fps)
-------------------------------------------------------------------
[구간 1]: 100.0s ~ 110.0s: 남성이 복도 바닥에 깔린 매트 위에서 누워 있으며, 움직임이 없어 정지 상태로 관찰됨.[정상]
